In [1]:
import requests
import json

# ================================
# Environment & Client
# ================================
import os
import aisuite as ai
from pathlib import Path
from dotenv import load_dotenv
_PREFERED_MODEL = "openai:gemini-3.5-flash-lite"
_REASONING_MODEL = "openai:gemini-3.8-flash"

env_path = Path("../.env.local")
load_dotenv(env_path)

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
# First API call with reasoning
response = requests.post(
  url="https://openrouter.ai/api/v1/chat/completions",
  headers={
    "Authorization": "Bearer <OPENROUTER_API_KEY>",
    "Content-Type": "application/json",
  },
  data=json.dumps({
    "model": "nvidia/nemotron-3-ultra-550b-a55b:free",
    "messages": [
        {
          "role": "user",
          "content": "How many r's are in the word 'strawberry'?"
        }
      ],
    "reasoning": {"enabled": True}
  })
)

# Extract the assistant message with reasoning_details
response = response.json()
print(response)  # Print the full response for debugging
#response = response['choices'][0]['message']
print(response)  # Print the assistant's message for debugging
"""

# Preserve the assistant message with reasoning_details
messages = [
  {"role": "user", "content": "How many r's are in the word 'strawberry'?"},
  {
    "role": "assistant",
    "content": response.get('content'),
    "reasoning_details": response.get('reasoning_details')  # Pass back unmodified
  },
  {"role": "user", "content": "Are you sure? Think carefully."}
]

# Second API call - model continues reasoning from where it left off
response2 = requests.post(
  url="https://openrouter.ai/api/v1/chat/completions",
  data=json.dumps({
    "model": "nvidia/nemotron-3-ultra-550b-a55b:free",
    "messages": messages,  # Includes preserved reasoning_details
    "reasoning": {"enabled": True}
  })
)
"""

{'error': {'message': 'Missing Authentication header', 'code': 401}}
{'error': {'message': 'Missing Authentication header', 'code': 401}}


'\n\n# Preserve the assistant message with reasoning_details\nmessages = [\n  {"role": "user", "content": "How many r\'s are in the word \'strawberry\'?"},\n  {\n    "role": "assistant",\n    "content": response.get(\'content\'),\n    "reasoning_details": response.get(\'reasoning_details\')  # Pass back unmodified\n  },\n  {"role": "user", "content": "Are you sure? Think carefully."}\n]\n\n# Second API call - model continues reasoning from where it left off\nresponse2 = requests.post(\n  url="https://openrouter.ai/api/v1/chat/completions",\n  data=json.dumps({\n    "model": "nvidia/nemotron-3-ultra-550b-a55b:free",\n    "messages": messages,  # Includes preserved reasoning_details\n    "reasoning": {"enabled": True}\n  })\n)\n'

In [2]:
from openai import OpenAI
from dotenv import load_dotenv
from pathlib import Path
import os
import json
# Localiza a raiz do projeto (diretório pai da subpasta)
env_path = Path("../.env.local")
load_dotenv(env_path)

api_key = os.getenv("OPENROUTER_API_KEY") or globals().get("OPENROUTER_API_KEY")

if not api_key:
    raise RuntimeError("OPENROUTER_API_KEY not found. Add it to .env.local or define it in a notebook cell.")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key
)

try:
    response = client.chat.completions.create(
        model="nvidia/nemotron-3-ultra-550b-a55b:free",
        messages=[
            {
                "role": "user",
                "content": "How many r's are in strawberry?"
            }
        ],
        extra_body={
            "reasoning": {"enabled": True}
        },
    )

    message = response.choices[0].message

    print("Resposta:", message.content)
    print("Raciocínio disponível:", getattr(message, "reasoning_details", None))
    print("Uso:", response.usage)

except Exception as e:
    print("Tipo do erro:", type(e).__name__)
    print("Detalhes:", str(e))

Resposta: There are **3** r's in strawberry.

Here is the breakdown: st**r**awbe**rr**y.
Raciocínio disponível: [{'type': 'reasoning.text', 'text': 'The user is asking for the count of the letter \'r\' in the word "strawberry".\nI need to count the occurrences of \'r\' in "strawberry".\nWord: s-t-r-a-w-b-e-r-r-y\nIndices:\n1: s\n2: t\n3: r (1)\n4: a\n5: w\n6: b\n7: e\n8: r (2)\n9: r (3)\n10: y\nTotal count is 3.', 'format': 'unknown', 'index': 0}]
Uso: CompletionUsage(completion_tokens=134, prompt_tokens=25, total_tokens=159, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=105, rejected_prediction_tokens=None, image_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0, cache_write_tokens=0, video_tokens=0), cost=0, is_byok=False, cost_details={'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0})


In [3]:
from ollama import chat

resposta = chat(
    model="deepseek-r1:8b",
    messages=[
        {
            "role": "user",
            "content": "Olá! Apresente-se e explique suas capacidades."
        }
    ]
)

print("\nResposta do DeepSeek-R1:\n")
print(resposta.message.content)


Resposta do DeepSeek-R1:


Olá! 😊 Sou a DeepSeek-R1, uma inteligência artificial desenvolvida para ser seu assistente confiável em questões acadêmicas, profissionais e criativas. Posso responder perguntas, explicar assuntos complexos, ajudar na escrita de textos, analisar dados, traduzir textos e muito mais — tudo baseado no meu conhecimento até julho de 2024.

Abaixo estão algumas das coisas que eu posso fazer:

### 💡 Capacidades

- **Explicação de conceitos**: Posso explicar ideias complexas de forma simples e clara, seja em matemática, ciência, tecnologia ou qualquer outro tema.
- **Escrita e criação**: Ajudo a escrever artigos, redigir e-mails, criar histórias, poemas, apresentações e textos criativos.
- **Tradução**: Faço traduções entre diversos idiomas com precisão.
- **Análise de dados e tabelas**: Se você me enviar dados numéricos, tabelas ou gráficos, eu posso ajudar a interpretá-los e resumir informações.
- **Estudo e pesquisa**: Auxilio no entendimento de textos acadêmicos

In [4]:
print(resposta.message)

role='assistant' content='\nOlá! 😊 Sou a DeepSeek-R1, uma inteligência artificial desenvolvida para ser seu assistente confiável em questões acadêmicas, profissionais e criativas. Posso responder perguntas, explicar assuntos complexos, ajudar na escrita de textos, analisar dados, traduzir textos e muito mais — tudo baseado no meu conhecimento até julho de 2024.\n\nAbaixo estão algumas das coisas que eu posso fazer:\n\n### 💡 Capacidades\n\n- **Explicação de conceitos**: Posso explicar ideias complexas de forma simples e clara, seja em matemática, ciência, tecnologia ou qualquer outro tema.\n- **Escrita e criação**: Ajudo a escrever artigos, redigir e-mails, criar histórias, poemas, apresentações e textos criativos.\n- **Tradução**: Faço traduções entre diversos idiomas com precisão.\n- **Análise de dados e tabelas**: Se você me enviar dados numéricos, tabelas ou gráficos, eu posso ajudar a interpretá-los e resumir informações.\n- **Estudo e pesquisa**: Auxilio no entendimento de textos 